# Areal Deprivation Index（ADI）作成手順書

---

## 1. 概要

本手順書は、2020年国勢調査の小地域集計を用いて

- **小地域レベルADI**
- **郵便番号レベルADI（面積按分）**

を作成するための手順を整理したものである。

本プロセスは以下の2つのNotebookで構成される。

- **Notebook 1**：ArealDeprivationIndex_smallarea.ipynb  
- **Notebook 2**：ArealDeprivationIndex_ziparea.ipynb  

---

## 2. 使用データ

### ■ 国勢調査（2020年 小地域集計）

使用する表：

- 第6表：世帯構成  
- 第7表：住宅  
- 第9表：労働力  
- 第12表：職業  

---

### ■ 空間データ

- 小地域ポリゴン（国勢調査2020）
- 郵便番号ポリゴン（GEOSPACE 2022）

---

## 3. 前処理ルール（重要）

### ■ 記号の扱い

| 記号 | 意味 | 処理 |
|------|------|------|
| "-" | 該当なし | **0に変換** |
| "X" | 秘匿 | **NaN（欠損）** |

---

### ■ 使用する行

- **男女 = 総数のみ使用**

---

## 4. 小地域レベルADI（Notebook 1）

---

### 4.1 final_unit_code の作成（最重要）

小地域データは以下の問題を含む：

- 秘匿地域
- 合算地域
- 階層構造（レベル2〜4）

これを統一するため、

👉 **final_unit_code を作成する**

---

#### ■ ルール

- 秘匿地域 → 秘匿先へ統合  
- 合算地域あり → その行を代表単位  
- それ以外 → 元コード  

👉 **Union-Find でグルーピング**

---

### 4.2 階層処理（keep_finest）

同じ地域に

- レベル2（町）
- レベル4（丁目）

が共存する場合：

👉 **細かい方だけ残す**

---

#### ■ 判定ロジック

- prefix一致で子コードが存在 → 粗い方は削除  
- 合算地域は例外的に保持  

---

### 4.3 コード正規化（重要）

#### ■ 統計側

- 町丁字コード → **6桁（右側0埋め）**

#### ■ ポリゴン側

- KEY_CODE → **11桁に右側0埋め**
- そこから
  - 市区町村コード（5桁）
  - 町丁字コード（6桁）
  を生成

👉 **ここがズレると merge 失敗するので最重要ポイント**

---

### 4.4 ADI構成変数

以下の8指標：


| 変数名 | 定義 |
|--------|------|
| old_couple_rate | 高齢夫婦のみ世帯 / 全世帯 |
| old_single_rate | 高齢単身世帯 / 全世帯 |
| single_parent_proxy_rate | ひとり親世帯（proxy）/ 全世帯 |
| rent_rate | 借家世帯 / 住宅世帯 |
| sales_service_rate | 販売・サービス職比率 |
| agriculture_rate | 農林漁業従事者比率 |
| blue_collar_rate | ブルーカラー職比率 |
| unemployment_rate | 失業率 |

---

### 4.5 ADI計算式

```python
ADI_raw =
  2.99 * old_couple_rate
+ 7.57 * old_single_rate
+ 17.4 * single_parent_proxy_rate
+ 2.22 * rent_rate
+ 4.03 * sales_service_rate
+ 6.05 * agriculture_rate
+ 5.38 * blue_collar_rate
+ 18.3 * unemployment_rate
```
---

### 4.6 非居住地域の扱い（重要）

```
exclude_nonresidential = (
    (総数 == 0) |
    (住宅に住む一般世帯 == 0)
)
```

---

#### ■ データの分岐

- **analysis版**
  - exclude_nonresidential == False
  - ADI_raw notna

- **full版**
  - すべて含む

---

👉 **ここは解析と可視化で意味が変わる重要ポイント**

---

### 4.7 出力

- 属性データ
- ポリゴン付きデータ（full / analysis）

---

## 5. 郵便番号レベルADI（Notebook 2）

---

### 5.1 基本アイデア

小地域を郵便番号に重ねて：

👉 **面積按分でADIを再計算**

---

### 5.2 面積按分

```
area_weight = intersect_area / smallarea_area
```

```
ADI_weighted = ADI_raw × area_weight
```

---

### 5.3 集約

```
ZIP_ADI = Σ(ADI_weighted) / Σ(area_weight)
```

---

### 5.4 注意点（重要）

- smallarea → zip に分割される  
- 重みの合計は **約1になるのが正常**

👉 **area_weight_sum ≈ 1 を必ず確認**

---

### 5.5 ポリゴン処理

- 郵便番号単位で dissolve
- ADIを結合

---

### 5.6 出力

- zip_adi_full
- zip_adi_analysis

---

## 6. 再現手順まとめ

---

### Notebook 1

1. 4表を読み込み
2. `"-"` を0、`"X"` を欠損として処理
3. `final_unit_code` を作成
4. `keep_finest` を作成
5. `keep_finest=True` の行だけ残す
6. `final_unit_code` 単位で集約
7. 4表を merge
8. ADIを計算
9. ポリゴン側に `final_key` を付与
10. `dissolve` して小地域ADI地理データを保存

---

### Notebook 2

1. `smallarea_adi_analysis.gpkg` を読む
2. 郵便番号ポリゴンを読む
3. intersection
4. 面積重み計算
5. 郵便番号単位に集約
6. 郵便番号ポリゴンを dissolve
7. full版とanalysis版を保存

---

## 7. 重要な落とし穴（実務上）

---

### ■ mergeできない原因のほぼ全て

👉 **コード不一致**

- 桁数違い
- 0埋め違い
- KEY_CODE vs S_AREA

---

### ■ ADI欠損の原因

👉 ほぼこれ：

- 分母が0（人口0）
- 非居住地域

---

### ■ 歯抜けの原因

👉 2パターン

- 統計に存在しない
- key不一致

---

## 8. まとめ

この手順により：

- 小地域ADI（高精度）
- 郵便番号ADI（応用用）

を一貫して作成できる。
